In [66]:
from pathlib import Path
import duckdb
import pandas as pd

In [67]:
rename_dict = {
    "Mã đơn hàng": "order_id",
    "Tên sản phẩm": "combo_name",
    "Tên phân loại hàng": "combo_variant_name",
    "Giá ưu đãi": "combo_promotion_price",
    "Số lượng": "combo_quantity",
    "Code": "product_code",
    "Tên SP": "product_name",
    "Giá": "product_price",
    "Số lượng": "product_quantity",
    "Thành tiền": "total_value"
}

In [68]:
# Thư mục chứa file đơn hàng 
folder_path = Path("samples-140526")

In [69]:
def read_excel(path, pattern, dict, sheetname=None, skiprow:int = None):
  dfs = []
  for f in path.glob(pattern):
    if sheetname:
      df = pd.read_excel(f, sheet_name=sheetname, skiprows=skiprow)
    else: 
      df = pd.read_excel(f, skiprows=skiprow) 

    df_rename = df.rename(columns=dict)
    
    select_cols = [c for c in dict.values() if c in df_rename.columns]   
    df_selectcols = df_rename[select_cols]
    df_selectcols['source_file'] = f.name
    
  dfs.append(df_selectcols)
  df_final = pd.concat(dfs, ignore_index=True)
  return df_final

In [70]:
# Đọc các file data gốc
order_raw_shopee = read_excel(folder_path, "order*.xlsx", rename_dict)

# Lấy kết quả trong file VBA
vba_result = read_excel(folder_path, "*.xlsm", rename_dict, sheetname="GHEPCODE", skiprow=5)
vba_order_ids = vba_result["order_id"].drop_duplicates()

# Lấy các order ID trong đơn hàng gốc
raw_order_ids = order_raw_shopee["order_id"]

In [71]:
# Kiểm tra xem 1 combo có thể có 2 giá
test_1 = duckdb.sql("""
    SELECT DISTINCT
    combo_name,
    combo_variant_name,
    combo_promotion_price,
    COUNT(DISTINCT combo_promotion_price) OVER (PARTITION BY combo_name, combo_variant_name) AS distinct_price
    FROM order_raw_shopee
    QUALIFY distinct_price > 1
""").to_df()

if len(test_1) > 0:
    print(f"Có {len(test_1)} trường hợp sau có nhiều mức giá")
    print(test_1)
else:
    print("Không có combo nào có nhiều mức giá")

Có 4 trường hợp sau có nhiều mức giá
                                          combo_name combo_variant_name  \
0  Sốt ướp thịt Hàn Quốc OFood gói 80g, giúp thị ...    Vị truyền thống   
1  [O’Food] Combo 4 rong biển giòn trộn 4 vị – Ăn...       Mix 4 vị 40g   
2  Sốt ướp thịt Hàn Quốc OFood gói 80g, giúp thị ...    Vị truyền thống   
3  [O’Food] Combo 4 rong biển giòn trộn 4 vị – Ăn...       Mix 4 vị 40g   

   combo_promotion_price  distinct_price  
0                11500.0               2  
1               157000.0               2  
2                 1000.0               2  
3               169000.0               2  


In [72]:
# Kiểm tra giữa file chạy và file order đã đủ đơn hàng chưa
test_2 = duckdb.sql(
    """
    WITH order_vba AS (
        SELECT DISTINCT order_id, source_file
        FROM vba_result
    ),
    order_raw AS (
        SELECT DISTINCT order_id, source_file 
        FROM order_raw_shopee
    )
    SELECT
    v.*,
    r.*
    FROM order_vba v
    FULL OUTER JOIN order_raw r USING (order_id)
    WHERE r.order_id IS NULL
""").to_df()

In [90]:
#  Trích xuất dữ liệu combo - chi tiết combo từ kết quả file VBA và file order gốc
test_3 = duckdb.sql("""
    SELECT
    v.order_id,
    r.combo_name AS combo_rname,
    r.combo_variant_name AS combo_variant_rname,
    v.product_code,
    v.product_name,
    v.product_price
    FROM vba_result v
    INNER JOIN order_raw_shopee r USING (order_id, combo_name, combo_variant_name)
    --WHERE r.order_id IS NOT NULL
""").to_df()

custom_filter = test_3['product_code'].isna()
test_3[custom_filter].drop_duplicates(subset=["combo_rname", "combo_variant_rname"])

,order_id,combo_rname,combo_variant_rname,product_code,product_name,product_price
0,260515ETQ0TP2W,"Combo 4 rong biển giòn trộn 2 vị cá hồi, rau&h...",4 trộn cá hồi 30g,NaN,NaN,172000.0
47,260515F4725TTE,[Tặng hộp hoặc sốt] Combo 3 gói sốt muối kim c...,4 cay đậm,NaN,NaN,100000.0
56,260515F5AQQABH,"Combo 4 rong biển giòn trộn 2 vị cá hồi, rau&h...",Mix 2 vị 30g,NaN,NaN,172000.0
68,260515FAU4K15V,[Tặng hộp hoặc sốt] Combo 3 gói sốt muối kim c...,4 cay dịu,NaN,NaN,100000.0
164,260515F51QRE0X,[Tặng 1 lốc 2 oliu] - Combo 3 hộp Snack rong c...,3 hộp 3 vị,NaN,NaN,270000.0
